In [1]:
import random
import torch
import os
import numpy as np
import pandas as pd
import polars as pl

In [2]:
INPUT_DIR = '.'

In [3]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [4]:
def make_aggregated_outputs(input_file, metrics = ['accuracy', 'precision', 'recall', 'f1', 'kappa', 'MCC']):
    dimensions = [
        'informational_vs_involved',
        'non-narrative_vs_narrative',
        'situation-dependent_vs_explicit',
        'non-persuasive_vs_persuasive',
        'non-abstract_vs_abstract',
        'compressed_vs_elaborated'
    ]

    saved_output_dict = {}

    for folder in os.listdir(INPUT_DIR):
        folder_path = os.path.join(INPUT_DIR, folder)

        if not (os.path.isdir(folder_path) and 'outputs' in folder):
            continue

        saved_output_dict[folder] = {
            dim: {metric: [] for metric in metrics}
            for dim in dimensions
        }

        for sub_folder in os.listdir(folder_path):
            sub_path = os.path.join(folder_path, sub_folder)

            if not (os.path.isdir(sub_path) and sub_folder.isdigit()):
                continue

            file_path = os.path.join(sub_path, f"{input_file}.csv")
            df = pd.read_csv(file_path)

            for dimension in dimensions:
                temp_df = df[df['dimension'] == dimension]

                if temp_df.empty:
                    raise ValueError(f"There should be something in the temp_df for the dimension {dimension}.")

                row = temp_df.iloc[0]

                for metric in metrics:
                    saved_output_dict[folder][dimension][metric].append(float(row[metric]))

    temp_dict_all = {}

    for folder, dimensions_dict in saved_output_dict.items():
        series_list = []

        for dimension, metrics_dict in dimensions_dict.items():
            mean_series = pd.Series({
                metric: (sum(values) / len(values)) if values else float('nan')
                for metric, values in metrics_dict.items()
            }, name=dimension)

            series_list.append(mean_series)

        df_folder = pd.concat(series_list, axis=1)
        temp_dict_all[folder] = df_folder

    df_all_folders = pd.concat(temp_dict_all, axis=0)

    df_mean_all = df_all_folders.groupby(level=1).mean()

    return df_all_folders, df_mean_all

In [5]:
all_classif, mean_classif = make_aggregated_outputs('classification_comparison_results_zero_vs_biber')

In [6]:
all_classif

informational_vs_involved  non-narrative_vs_narrative  \
outputsTrain accuracy                    0.628300                    0.571900   
             precision                   0.538970                    0.474405   
             recall                      0.710152                    0.573796   
             f1                          0.611926                    0.518102   
             kappa                       0.268272                    0.139959   
             MCC                         0.278347                    0.142310   
outputsTest  accuracy                    0.628600                    0.555000   
             precision                   0.538734                    0.451563   
             recall                      0.695584                    0.553112   
             f1                          0.606423                    0.496026   
             kappa                       0.265601                    0.104863   
             MCC                         0.273906                    0.107107   
outputsAll   accuracy                    0.630300                    0.560400   
             precision                   0.541768                    0.462168   
             recall                      0.705986                    0.565050   
             f1                          0.612038                    0.506893   
             kappa                       0.270811                    0.118381   
             MCC                         0.280211                    0.120833   

                        situation-dependent_vs_explicit  \
outputsTrain accuracy                          0.611700   
             precision                         0.666973   
             recall                            0.657700   
             f1                                0.661534   
             kappa                             0.205375   
             MCC                               0.206009   
outputsTest  accuracy                          0.610200   
             precision                         0.668423   
             recall                            0.647514   
             f1                                0.656975   
             kappa                             0.205312   
             MCC                               0.206124   
outputsAll   accuracy                          0.614600   
             precision                         0.670434   
             recall                            0.655650   
             f1                                0.662047   
             kappa                             0.213474   
             MCC                               0.214289   

                        non-persuasive_vs_persuasive  \
outputsTrain accuracy                       0.561600   
             precision                      0.410626   
             recall                         0.567088   
             f1                             0.475110   
             kappa                          0.115240   
             MCC                            0.120175   
outputsTest  accuracy                       0.554200   
             precision                      0.407902   
             recall                         0.556685   
             f1                             0.469370   
             kappa                          0.100916   
             MCC                            0.105073   
outputsAll   accuracy                       0.556000   
             precision                      0.401693   
             recall                         0.556422   
             f1                             0.465037   
             kappa                          0.103147   
             MCC                            0.107815   

                        non-abstract_vs_abstract  compressed_vs_elaborated  
outputsTrain accuracy                   0.576800                  0.440900  
             precision                  0.285432                  0.125185  
             recall                     0.637540                  

In [7]:
mean_classif

,informational_vs_involved,non-narrative_vs_narrative,situation-dependent_vs_explicit,non-persuasive_vs_persuasive,non-abstract_vs_abstract,compressed_vs_elaborated
MCC,0.277488,0.123417,0.208807,0.111021,0.157367,-0.156565
accuracy,0.629067,0.562433,0.612167,0.557267,0.574433,0.440767
f1,0.610129,0.507007,0.660185,0.469839,0.389874,0.181288
kappa,0.268228,0.121068,0.208054,0.106434,0.132039,-0.122234
precision,0.539824,0.462712,0.668610,0.406741,0.283646,0.125386
recall,0.703907,0.563986,0.653621,0.560065,0.632566,0.333246


In [8]:
all_contin, mean_contin = make_aggregated_outputs('continuous_comparison_results_zero_vs_biber', ['pearson', 'spearman', 'MSE', 'RMSE', 'MAE'])

In [9]:
all_contin

informational_vs_involved  non-narrative_vs_narrative  \
outputsTrain pearson                    0.365088                    0.092122   
             spearman                   0.368936                    0.154543   
             MSE                        1.269824                    1.815756   
             RMSE                       1.124847                    1.345978   
             MAE                        0.913504                    1.010720   
outputsTest  pearson                    0.344727                    0.063803   
             spearman                   0.350697                    0.114826   
             MSE                        1.310547                    1.872395   
             RMSE                       1.143203                    1.366621   
             MAE                        0.921203                    1.032493   
outputsAll   pearson                    0.361515                    0.077538   
             spearman                   0.368417                    0.140168   
             MSE                        1.276971                    1.844924   
             RMSE                       1.128212                    1.356706   
             MAE                        0.912374                    1.020831   

                       situation-dependent_vs_explicit  \
outputsTrain pearson                          0.157739   
             spearman                         0.251102   
             MSE                              1.684522   
             RMSE                             1.296024   
             MAE                              0.974766   
outputsTest  pearson                          0.144628   
             spearman                         0.247145   
             MSE                              1.710743   
             RMSE                             1.305688   
             MAE                              0.984844   
outputsAll   pearson                          0.167724   
             spearman                         0.270603   
             MSE                              1.664552   
             RMSE                             1.287657   
             MAE                              0.965608   

                       non-persuasive_vs_persuasive  non-abstract_vs_abstract  \
outputsTrain pearson                       0.118807                  0.125781   
             spearman                      0.163040                  0.205356   
             MSE                           1.762386                  1.748438   
             RMSE                          1.325851                  1.319869   
             MAE                           0.987530                  0.978905   
outputsTest  pearson                       0.121981                  0.115263   
             spearman                      0.157591                  0.190817   
             MSE                           1.756038                  1.769474   
             RMSE                          1.323102                  1.327908   
             MAE                           0.991895                  0.986914   
outputsAll   pearson                       0.127700                  0.113505   
             spearman                      0.160640                  0.198806   
             MSE                           1.744601                  1.772989   
             RMSE                          1.318993                  1.329257   
             MAE                           0.987891                  0.982819   

                       compressed_vs_elaborated  
outputsTrain pearson                  -0.066467  
             spearman                 -0.174349  
             MSE                       2.132934  
             RMSE                      1.458751  
             MAE                       1.101461  
outputsTest  pearson                  -0.063735  
             spearman                 -0.178484  
             MSE                       2.127470  
             RMSE                      1.457391  
             MAE

In [10]:
mean_contin

,informational_vs_involved,non-narrative_vs_narrative,situation-dependent_vs_explicit,non-persuasive_vs_persuasive,non-abstract_vs_abstract,compressed_vs_elaborated
MAE,0.915694,1.021348,0.975073,0.989106,0.982879,1.107157
MSE,1.285780,1.844358,1.686606,1.754342,1.763634,2.132208
RMSE,1.132088,1.356435,1.296456,1.322648,1.325678,1.458770
pearson,0.357110,0.077821,0.156697,0.122829,0.118183,-0.066104
spearman,0.362683,0.136512,0.256283,0.160424,0.198326,-0.173657
